# LSTM-KAN Time Series Forecasting (Colab Ready)

This notebook is the simple path for running the project in Google Colab or local Jupyter. It trains one selected model by default. Change `MODEL_NAMES` to `['GRU', 'GRUKAN', 'LSTM', 'LSTMKAN']` when you want a full comparison.

In [ ]:
# Bootstrap repo path. In Colab this clones the GitHub repository once.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Wayan123/LSTM-KAN-for-Time-Series.git"

if "google.colab" in sys.modules:
    repo_dir = Path("/content/LSTM-KAN-for-Time-Series")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
else:
    repo_dir = Path.cwd()
    if not (repo_dir / "lstm_kan").exists() and (repo_dir.parent / "lstm_kan").exists():
        repo_dir = repo_dir.parent
    os.chdir(repo_dir)

sys.path.insert(0, str(repo_dir))
print(f"Repository: {repo_dir}")


In [ ]:
# Install only missing notebook dependencies.
import importlib.util
import subprocess
import sys

required = ["numpy", "pandas", "matplotlib", "torch"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from lstm_kan.data import load_energy_dataset
from lstm_kan.inference import predict_from_csv
from lstm_kan.models import build_model
from lstm_kan.training import build_loaders, evaluate_by_file, fit_model, save_artifact, set_seed

In [ ]:
# Experiment settings. Keep MAX_FILES small for a quick first run.
DATA_DIR = Path("dataset")
ARTIFACT_ROOT = Path("artifacts/colab")
MODEL_NAMES = ["LSTMKAN"]  # Try ["GRU", "GRUKAN", "LSTM", "LSTMKAN"] for a full comparison.
MAX_FILES = 1
WINDOW_SIZE = 90
HIDDEN_DIM = 128
N_LAYERS = 2
DROPOUT = 0.2
EPOCHS = 5
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
PATIENCE = 3
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
set_seed(SEED)

dataset = load_energy_dataset(
    DATA_DIR,
    window_size=WINDOW_SIZE,
    max_files=MAX_FILES,
    train_ratio=0.7,
    val_ratio=0.15,
)

print("train_x", dataset.train_x.shape, "train_y", dataset.train_y.shape)
print("val_x", dataset.val_x.shape, "val_y", dataset.val_y.shape)
pd.DataFrame([summary.__dict__ for summary in dataset.file_summaries])

In [ ]:
train_loader, val_loader = build_loaders(dataset, BATCH_SIZE)
artifacts = {}
histories = {}
trained_models = {}

for model_name in MODEL_NAMES:
    print(f"\n=== Training {model_name} ===")
    model = build_model(
        model_name,
        input_dim=len(dataset.feature_columns),
        hidden_dim=HIDDEN_DIM,
        output_dim=1,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    )
    model, history = fit_model(
        model,
        train_loader,
        val_loader,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        patience=PATIENCE,
        device=DEVICE,
        target_scaler=dataset.target_scaler,
        verbose=True,
    )
    artifact_dir = ARTIFACT_ROOT / model_name
    save_artifact(
        model,
        artifact_dir,
        model_name=model_name,
        model_kwargs={
            "hidden_dim": HIDDEN_DIM,
            "n_layers": N_LAYERS,
            "dropout": DROPOUT,
            "output_dim": 1,
        },
        feature_columns=dataset.feature_columns,
        target_column=dataset.target_column,
        window_size=WINDOW_SIZE,
        feature_scaler=dataset.feature_scaler,
        target_scaler=dataset.target_scaler,
        history=history,
        extra_metadata={"source": "colab_notebook", "max_files": MAX_FILES, "seed": SEED},
    )
    artifacts[model_name] = artifact_dir
    histories[model_name] = history

In [ ]:
# Evaluate the true held-out test windows created by the leakage-free splitter.
evaluation_rows = []
for model_name, model in trained_models.items():
    rows = evaluate_by_file(
        model,
        dataset.test_x_by_file,
        dataset.test_y_by_file,
        target_scaler=dataset.target_scaler,
        device=DEVICE,
        batch_size=BATCH_SIZE,
    )
    for row in rows:
        evaluation_rows.append({"model": model_name, **row})

metrics_df = pd.DataFrame(evaluation_rows)
metrics_df.sort_values(["smape", "rmse"])


In [ ]:
# Plot the latest predictions for the first trained model and first file.
model_name = MODEL_NAMES[0]
selected_files = sorted(DATA_DIR.glob("*.csv"))[:MAX_FILES]
sample_file = selected_files[0]
result = predict_from_csv(artifacts[model_name], sample_file, device=DEVICE, batch_size=BATCH_SIZE)
plot_df = result["predictions"].tail(300)

plt.figure(figsize=(14, 5))
plt.plot(plot_df["Datetime"], plot_df["actual"], label="Actual", linewidth=2)
plt.plot(plot_df["Datetime"], plot_df["predicted"], label=f"{model_name} prediction", linewidth=2)
plt.title(f"{model_name} prediction on {sample_file.name}")
plt.xlabel("Datetime")
plt.ylabel("Energy consumption (MW)")
plt.legend()
plt.tight_layout()
plt.show()

## Local deployment after training

After saving artifacts, the same model can be used outside the notebook:

```bash
python train_local.py --models LSTMKAN --copy-latest
streamlit run streamlit_app.py
uvicorn api:app --host 127.0.0.1 --port 8000
node dashboard-node/server.js
```